# 01 - Extraccion: Ingesta desde PubChem API hacia Capa Raw
## Fuente: API PUG REST de PubChem (NIH)

---

### Contexto de la fuente de datos

**PubChem** es la base de datos publica de quimica mas grande del mundo, mantenida por el
**NIH** (Instituto Nacional de Salud de EE.UU.) a traves de la Biblioteca Nacional de Medicina.
Contiene informacion sobre millones de compuestos quimicos, sus propiedades fisicoquimicas
y los resultados de ensayos biologicos (bioassays).

### Que datos extraemos?

Para cada uno de los 15 medicamentos seleccionados, extraemos dos tipos de informacion:

1. **Propiedades fisicoquimicas del compuesto**: formula molecular, peso molecular,
   coeficiente de particion (XLogP), area de superficie polar topologica (TPSA),
   complejidad molecular, conteo de donantes/aceptores de enlaces de hidrogeno, etc.

2. **Resultados de bioensayos**: registros de pruebas de laboratorio que determinan
   si un compuesto es activo o inactivo contra un objetivo biologico especifico
   (gen, proteina, celula, organismo).

### Por que esta fuente?

- Pertenece al ambito de **farmacologia y ciencias de la salud**
- API **publica y gratuita** del NIH (no requiere autenticacion ni API key)
- Datos **complejos** con multiples relaciones (compuesto <-> ensayo <-> resultado)
- Permite construir un **modelo estrella** con dimensiones naturales
- Accesible desde **Databricks Serverless** (dominio nih.gov permitido por firewall)

### Endpoint del API

Utilizamos el servicio **PUG REST** (Power User Gateway):
- URL base: `https://pubchem.ncbi.nlm.nih.gov/rest/pug`
- Documentacion oficial: https://pubchem.ncbi.nlm.nih.gov/docs/pug-rest

---
## 1. Importacion de Librerias

Utilizamos unicamente librerias de la **biblioteca estandar de Python**,
lo cual garantiza compatibilidad con cualquier entorno de Databricks
sin necesidad de instalar paquetes adicionales.

In [ ]:
import urllib.request   # Para realizar peticiones HTTP al API de PubChem
import ssl              # Para manejar el contexto SSL de las conexiones HTTPS
import json             # Para serializar y deserializar datos en formato JSON
import os               # Para operaciones con el sistema de archivos (tamano de archivos)
import time             # Para implementar pausas entre peticiones al API (rate limiting)
from datetime import datetime  # Para generar marcas de tiempo en los archivos

---
## 2. Configuracion de la Extraccion

Definimos todos los parametros necesarios para la extraccion:

- **URL base** del API PUG REST
- **Ruta de destino** en el volumen de Unity Catalog
- **Diccionario de medicamentos** con sus CIDs (PubChem Compound IDs)
- **Contexto SSL** para las conexiones HTTPS

Los 15 medicamentos fueron seleccionados para cubrir diversas **categorias terapeuticas**:
analgesicos, antiinflamatorios, ansioliticos, antidepresivos, antibioticos, etc.

In [ ]:
# URL base de la API PUG REST de PubChem (servicio del NIH)
URL_BASE = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"

# Detectar catalogo activo y construir ruta del volumen de destino
# La ruta sigue el formato de volumenes de Unity Catalog
CATALOGO = spark.sql("SELECT current_catalog()").collect()[0][0]
RUTA_RAW = f"/Volumes/{CATALOGO}/pubchem_raw/bioactividad"

# Diccionario de medicamentos a extraer
# Clave: CID (PubChem Compound ID) - identificador unico del compuesto en PubChem
# Valor: Nombre comun del medicamento en espanol
# Se seleccionaron medicamentos de diferentes categorias terapeuticas para diversidad
MEDICAMENTOS = {
    2244: "Aspirina",           # Analgesico / Antiinflamatorio no esteroideo (AINE)
    3672: "Ibuprofeno",         # Antiinflamatorio no esteroideo (AINE)
    2662: "Celecoxib",          # Antiinflamatorio selectivo COX-2
    5770: "Paracetamol",        # Analgesico / Antipiretico (Acetaminofen)
    5329: "Naproxeno",          # Antiinflamatorio no esteroideo (AINE)
    2719: "Diazepam",           # Ansiolitico / Benzodiacepina
    5743: "Morfina",            # Analgesico opioide potente
    2519: "Cafeina",            # Estimulante del sistema nervioso central (SNC)
    3345: "Fluoxetina",         # Antidepresivo inhibidor selectivo de recaptacion de serotonina (ISRS)
    4091: "Metformina",         # Antidiabetico oral de primera linea
    2082: "Amoxicilina",        # Antibiotico betalactamico de amplio espectro
    5494: "Omeprazol",          # Inhibidor de la bomba de protones (IBP)
    5311: "Melatonina",         # Regulador del ciclo sueno-vigilia
    4386: "Nicotina",           # Estimulante / Agonista de receptores nicotinicos
    5640: "Loratadina",         # Antihistaminico de segunda generacion
}

# Contexto SSL para las peticiones HTTPS al API
# Necesario para verificar certificados del servidor del NIH
CONTEXTO_SSL = ssl.create_default_context()

# Mostrar resumen de configuracion
print(f"Medicamentos a extraer: {len(MEDICAMENTOS)}")
print(f"Ruta de destino: {RUTA_RAW}")
print(f"URL base del API: {URL_BASE}")

---
## 3. Funciones de Extraccion

Se definen tres funciones modulares que encapsulan la logica de comunicacion con el API:

1. **`llamar_api_pubchem()`**: Funcion generica para hacer peticiones GET al API
2. **`extraer_propiedades_compuesto()`**: Obtiene propiedades fisicoquimicas de un compuesto
3. **`extraer_bioactividad_compuesto()`**: Obtiene resultados de bioensayos de un compuesto

Se implementa manejo de errores robusto para que un fallo en un compuesto
no detenga la extraccion de los demas.

In [ ]:
def llamar_api_pubchem(url, timeout=30):
    """
    Realiza una peticion GET a la API PUG REST de PubChem.

    Esta funcion encapsula la logica de comunicacion HTTP con el servidor
    del NIH, incluyendo manejo de SSL, headers y timeout.

    Parametros
    ----------
    url : str
        URL completa del endpoint del API a consultar.
    timeout : int, opcional
        Tiempo maximo de espera en segundos (por defecto 30).

    Retorna
    -------
    dict o None
        Diccionario con la respuesta JSON parseada, o None si ocurre un error.
    """
    try:
        # Crear la peticion con un User-Agent personalizado
        # Algunos APIs rechazan peticiones sin User-Agent valido
        peticion = urllib.request.Request(
            url,
            headers={"User-Agent": "Mozilla/5.0 (Databricks)"}
        )

        # Ejecutar la peticion HTTPS con contexto SSL y timeout
        with urllib.request.urlopen(peticion, timeout=timeout, context=CONTEXTO_SSL) as respuesta:
            # Leer la respuesta, decodificar de bytes a string y parsear el JSON
            contenido = respuesta.read().decode("utf-8")
            return json.loads(contenido)

    except Exception as error:
        # Capturar cualquier error (timeout, conexion, HTTP 404/500, etc.)
        # Truncar el mensaje a 80 caracteres para legibilidad
        print(f"    Error en peticion al API: {str(error)[:80]}")
        return None


def extraer_propiedades_compuesto(cid):
    """
    Extrae las propiedades fisicoquimicas de un compuesto dado su CID.

    Consulta el endpoint de propiedades del API PUG REST para obtener
    formula molecular, peso, solubilidad (XLogP), complejidad, etc.

    Parametros
    ----------
    cid : int
        PubChem Compound ID del compuesto a consultar.

    Retorna
    -------
    dict o None
        Diccionario con las propiedades del compuesto, o None si hay error.
    """
    # Definir los campos de propiedades a solicitar al API
    campos = (
        "MolecularFormula,MolecularWeight,IUPACName,XLogP,TPSA,"
        "Complexity,HBondDonorCount,HBondAcceptorCount,"
        "RotatableBondCount,HeavyAtomCount"
    )

    # Construir la URL del endpoint de propiedades
    url = f"{URL_BASE}/compound/cid/{cid}/property/{campos}/JSON"

    # Realizar la peticion al API
    datos = llamar_api_pubchem(url)

    # Validar que la respuesta contenga la estructura esperada
    if datos and "PropertyTable" in datos:
        # Retornar el primer (y unico) registro de propiedades
        return datos["PropertyTable"]["Properties"][0]

    return None


def extraer_bioactividad_compuesto(cid):
    """
    Extrae los resultados de bioensayos de un compuesto dado su CID.

    Cada registro representa un resultado individual: en que ensayo se probo
    el compuesto, si fue activo/inactivo, el valor de actividad medido, etc.
    Un medicamento puede tener miles de resultados de bioensayos.

    Parametros
    ----------
    cid : int
        PubChem Compound ID del compuesto a consultar.

    Retorna
    -------
    list[dict]
        Lista de diccionarios, cada uno representando un resultado de bioensayo.
        Retorna lista vacia si hay error o no hay datos.
    """
    # Endpoint de resumen de ensayos para un compuesto especifico
    url = f"{URL_BASE}/compound/cid/{cid}/assaysummary/JSON"

    # Timeout mayor (60s) porque esta consulta puede retornar grandes volumenes
    datos = llamar_api_pubchem(url, timeout=60)

    # Validar estructura de la respuesta
    if datos and "Table" in datos:
        # Extraer nombres de columnas y filas de datos
        columnas = datos["Table"]["Columns"]["Column"]
        filas = datos["Table"]["Row"]

        # Convertir la estructura tabular del API a lista de diccionarios
        # Cada fila se transforma en un diccionario {columna: valor}
        registros = []
        for fila in filas:
            registro = {}
            for i, col in enumerate(columnas):
                # Asignar el valor de la celda, o cadena vacia si no existe
                registro[col] = fila["Cell"][i] if i < len(fila["Cell"]) else ""
            registros.append(registro)

        return registros

    return []

---
## 4. Ejecucion de la Extraccion Completa

Se itera sobre los 15 medicamentos y para cada uno se extraen:

1. **Propiedades fisicoquimicas** (1 registro por medicamento)
2. **Resultados de bioensayos** (cientos a miles de registros por medicamento)

Se incluye una **pausa de 0.5 segundos** entre cada medicamento para respetar
los limites de uso del API del NIH (rate limiting) y evitar que nos bloqueen.

Al finalizar, se muestra un resumen con el total de datos extraidos.

In [ ]:
# Generar marca de tiempo unica para los archivos de esta ejecucion
# Formato: YYYYMMDD_HHMMSS para facilitar ordenamiento cronologico
marca_tiempo = datetime.now().strftime("%Y%m%d_%H%M%S")

# Listas acumuladoras para almacenar todos los datos extraidos
todas_las_propiedades = []   # Almacenara las propiedades de los 15 compuestos
todas_las_actividades = []   # Almacenara todos los registros de bioactividad

# Registrar inicio de la extraccion
print(f"Inicio de extraccion: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Fuente: {URL_BASE}")
print("-" * 65)

# Iterar sobre cada medicamento del diccionario
for cid, nombre in MEDICAMENTOS.items():
    print(f"  Extrayendo {nombre} (CID {cid})...")

    # Paso 1: Extraer propiedades fisicoquimicas del compuesto
    propiedades = extraer_propiedades_compuesto(cid)
    if propiedades:
        # Agregar el nombre comun del medicamento al registro de propiedades
        propiedades["nombre_comun"] = nombre
        todas_las_propiedades.append(propiedades)

    # Paso 2: Extraer resultados de bioensayos
    actividades = extraer_bioactividad_compuesto(cid)
    # Agregar el nombre comun a cada registro de actividad para trazabilidad
    for actividad in actividades:
        actividad["nombre_comun"] = nombre
    todas_las_actividades.extend(actividades)

    # Mostrar estado de la extraccion para este medicamento
    estado_props = "OK" if propiedades else "Error"
    print(f"    Propiedades: {estado_props} | Actividades: {len(actividades)}")

    # Pausa de 0.5 segundos para respetar los limites del API del NIH
    time.sleep(0.5)

# Resumen final de la extraccion
print("-" * 65)
print(f"Extraccion finalizada: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Compuestos extraidos: {len(todas_las_propiedades)}")
print(f"Registros de bioactividad: {len(todas_las_actividades)}")

---
## 5. Guardar Datos Crudos en Formato JSON (Capa Raw)

Los datos extraidos se guardan como archivos JSON en el volumen de Unity Catalog.
Se generan dos archivos con marca de tiempo en el nombre:

1. `propiedades_YYYYMMDD_HHMMSS.json` - Propiedades fisicoquimicas de cada compuesto
2. `bioactividad_YYYYMMDD_HHMMSS.json` - Resultados de bioensayos

La marca de tiempo en el nombre permite mantener un **historial de ingestas**
y facilita la identificacion del archivo mas reciente en notebooks posteriores.

In [ ]:
# Guardar archivo JSON de propiedades de compuestos
archivo_props = f"{RUTA_RAW}/propiedades_{marca_tiempo}.json"
with open(archivo_props, "w", encoding="utf-8") as archivo:
    # ensure_ascii=False permite guardar caracteres especiales sin escapar
    json.dump(todas_las_propiedades, archivo, ensure_ascii=False)

# Calcular tamano del archivo en megabytes
tamano_props = os.path.getsize(archivo_props) / (1024 * 1024)
print(f"Propiedades guardadas: {archivo_props}")
print(f"  Registros: {len(todas_las_propiedades)} | Tamano: {tamano_props:.2f} MB")

# Guardar archivo JSON de bioactividad
archivo_bio = f"{RUTA_RAW}/bioactividad_{marca_tiempo}.json"
with open(archivo_bio, "w", encoding="utf-8") as archivo:
    json.dump(todas_las_actividades, archivo, ensure_ascii=False)

# Calcular tamano del archivo en megabytes
tamano_bio = os.path.getsize(archivo_bio) / (1024 * 1024)
print(f"\nBioactividad guardada: {archivo_bio}")
print(f"  Registros: {len(todas_las_actividades)} | Tamano: {tamano_bio:.2f} MB")

---
## 6. Verificacion y Registro de la Extraccion

Se realiza una **verificacion de integridad** leyendo los archivos guardados
y comparando el numero de registros con los datos en memoria.
Esto asegura que la escritura al disco fue completa y sin corrupcion.

Finalmente se muestra un registro consolidado de la extraccion.

In [ ]:
# Verificar integridad leyendo los archivos guardados
with open(archivo_props, "r") as archivo:
    verificacion_props = json.load(archivo)
with open(archivo_bio, "r") as archivo:
    verificacion_bio = json.load(archivo)

# Validar que la cantidad de registros coincida con los datos en memoria
assert len(verificacion_props) == len(todas_las_propiedades), \
    "Error: la cantidad de registros de propiedades no coincide"
assert len(verificacion_bio) == len(todas_las_actividades), \
    "Error: la cantidad de registros de bioactividad no coincide"

# Mostrar registro consolidado de la extraccion
print("=" * 55)
print("  REGISTRO DE EXTRACCION")
print("=" * 55)
print(f"  Fuente:          PubChem PUG REST API (NIH)")
print(f"  Marca de tiempo: {marca_tiempo}")
print(f"  Compuestos:      {len(todas_las_propiedades)}")
print(f"  Actividades:     {len(todas_las_actividades):,}")
print(f"  Tamano total:    {tamano_props + tamano_bio:.2f} MB")
print(f"  Estado:          EXITOSO")
print("=" * 55)